### Traiter au niveau du corpus

In [ ]:
import glob
from transformers import pipeline, TokenClassificationPipeline, CamembertTokenizer
from itertools import chain

# === Initialisation du modèle ===
model_name = "Jean-Baptiste/camembert-ner-with-dates"
tokenizer = CamembertTokenizer.from_pretrained(model_name)
nlp: TokenClassificationPipeline = pipeline(
    "ner",
    model=model_name,
    tokenizer=tokenizer,
    aggregation_strategy="simple",
    device=-1
)

# === Fonction de découpage ===
def chunk_text(text, chunk_size=512, overlap=50):
    start = 0
    while start < len(text):
        end = start + chunk_size
        yield text[start:end]
        start += chunk_size - overlap

# === Extraction d'entités sur un grand texte ===
def extraire_entites(texte, nlp):
    chunks = list(chunk_text(texte))
    all_ner_results = list(chain.from_iterable([nlp(chunk) for chunk in chunks]))
    return [
        {"mot": ent["word"], "type": ent["entity_group"]}
        for ent in all_ner_results
    ]

# === Étape 1 : concaténer tout le corpus en une seule chaîne ===
corpus_global = ""

for path_fichier in glob.glob("sample_data/Corpus/*.txt"):
    with open(path_fichier, "r", encoding="utf-8") as f:
        texte = f.read().strip()
        corpus_global += texte + "\n"  # Ajouter un retour à la ligne entre les fichiers

print("Tous les fichiers ont été fusionnés dans un seul texte.")
print(f"Taille totale du corpus (caractères) : {len(corpus_global)}")

# === Étape 2 : extraction d'entités sur l'ensemble du corpus ===
entites_globales = extraire_entites(corpus_global, nlp)
tokens = tokenizer.tokenize(corpus_global)
print(f"Nombre total de tokens (subwords) : {len(tokens)}")
print(f"Nombre total d'entités : {len(entites_globales)}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/423 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/811k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/210 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/970 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Device set to use cpu


Tous les fichiers ont été fusionnés dans un seul texte.
Taille totale du corpus (caractères) : 1364886


In [ ]:
import json
resultat = {"entites": entites_globales}
with open("tous_entites_nommes_Camembert.json", "w", encoding="utf-8") as f:
    json.dump(resultat, f, indent=4, ensure_ascii=False)

去重

In [2]:
import glob
from transformers import pipeline, TokenClassificationPipeline, CamembertTokenizer
from itertools import chain

# === Initialisation du modèle ===
model_name = "Jean-Baptiste/camembert-ner-with-dates"
tokenizer = CamembertTokenizer.from_pretrained(model_name)
nlp: TokenClassificationPipeline = pipeline(
    "ner",
    model=model_name,
    tokenizer=tokenizer,
    aggregation_strategy="simple",
    device=-1
)

# === Fonction de découpage ===
def chunk_text(text, chunk_size=512, overlap=50):
    start = 0
    while start < len(text):
        end = start + chunk_size
        yield text[start:end]
        start += chunk_size - overlap

# === Extraction d'entités sur un grand texte avec option de déduplication ===
def extraire_entites(texte, nlp, unique=False):
    chunks = list(chunk_text(texte))
    all_ner_results = list(chain.from_iterable([nlp(chunk) for chunk in chunks]))

    entites = [
        {"mot": ent["word"], "type": ent["entity_group"]}
        for ent in all_ner_results
    ]

    if unique:  # 去重
        seen = set()
        entites_uniques = []
        for ent in entites:
            key = (ent["mot"], ent["type"])
            if key not in seen:
                entites_uniques.append(ent)
                seen.add(key)
        return entites_uniques

    return entites

# === Étape 1 : concaténer tout le corpus en une seule chaîne ===
corpus_global = ""

for path_fichier in glob.glob("sample_data/Corpus/*.txt"):
    with open(path_fichier, "r", encoding="utf-8") as f:
        texte = f.read().strip()
        corpus_global += texte + "\n"  # Ajouter un retour à la ligne entre les fichiers

print("Tous les fichiers ont été fusionnés dans un seul texte.")
print(f"Taille totale du corpus (caractères) : {len(corpus_global)}")

# === Étape 2 : extraction d'entités sur l'ensemble du corpus ===
# ⚠️ 设置 unique=True 才会去重
entites_globales = extraire_entites(corpus_global, nlp, unique=True)

tokens = tokenizer.tokenize(corpus_global)
print(f"Nombre total de tokens (subwords) : {len(tokens)}")
print(f"Nombre total d'entités : {len(entites_globales)}")


Device set to use cpu


Tous les fichiers ont été fusionnés dans un seul texte.
Taille totale du corpus (caractères) : 1364886
Nombre total de tokens (subwords) : 375158
Nombre total d'entités : 5115


In [3]:
import glob
import time
from transformers import pipeline, TokenClassificationPipeline, CamembertTokenizer
from itertools import chain

# === Initialisation du modèle ===
model_name = "Jean-Baptiste/camembert-ner-with-dates"
tokenizer = CamembertTokenizer.from_pretrained(model_name)
nlp: TokenClassificationPipeline = pipeline(
    "ner",
    model=model_name,
    tokenizer=tokenizer,
    aggregation_strategy="simple",
    device=-1
)

# === Fonction de découpage ===
def chunk_text(text, chunk_size=512, overlap=50):
    start = 0
    while start < len(text):
        end = start + chunk_size
        yield text[start:end]
        start += chunk_size - overlap

# === Extraction d'entités sur un grand texte ===
def extraire_entites(texte, nlp):
    chunks = list(chunk_text(texte))
    all_ner_results = list(chain.from_iterable([nlp(chunk) for chunk in chunks]))
    return [
        {"mot": ent["word"], "type": ent["entity_group"]}
        for ent in all_ner_results
    ]

# === Étape 1 : concaténer tout le corpus en une seule chaîne ===
start_concat = time.time()
corpus_global = ""

for path_fichier in glob.glob("sample_data/Corpus/*.txt"):
    with open(path_fichier, "r", encoding="utf-8") as f:
        texte = f.read().strip()
        corpus_global += texte + "\n"  # Ajouter un retour à la ligne entre les fichiers

end_concat = time.time()
print("Tous les fichiers ont été fusionnés dans un seul texte.")
print(f"Taille totale du corpus (caractères) : {len(corpus_global)}")
print(f"Temps pour la concaténation : {end_concat - start_concat:.2f} secondes")

# === Étape 2 : extraction d'entités sur l'ensemble du corpus ===
start_ner = time.time()
entites_globales = extraire_entites(corpus_global, nlp)
end_ner = time.time()

tokens = tokenizer.tokenize(corpus_global)
print(f"Nombre total de tokens (subwords) : {len(tokens)}")
print(f"Nombre total d'entités : {len(entites_globales)}")
print(f"Temps pour l'extraction d'entités : {end_ner - start_ner:.2f} secondes")

# === Temps total ===
print(f"Temps total du script : {end_ner - start_concat:.2f} secondes")


Device set to use cpu


Tous les fichiers ont été fusionnés dans un seul texte.
Taille totale du corpus (caractères) : 1364886
Temps pour la concaténation : 0.03 secondes
Nombre total de tokens (subwords) : 375158
Nombre total d'entités : 9184
Temps pour l'extraction d'entités : 1818.51 secondes
Temps total du script : 1818.54 secondes


In [6]:
!pip install stanza


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 590.6/590.6 kB 28.0 MB/s eta 0:00:00


In [7]:
import stanza
def load_stanza_model(lang: str = "fr") -> stanza.Pipeline:
    try:
        nlp = stanza.Pipeline(lang=lang, processors='tokenize,ner')
    except:
        stanza.download(lang=lang, logging_level='DEBUG')
        nlp = stanza.Pipeline(lang=lang, processors='tokenize,ner')
    return nlp

In [16]:

import stanza
import re
import glob
import time
import json

def load_stanza_model(lang: str = "fr") -> stanza.Pipeline:
    return stanza.Pipeline(lang=lang, processors='tokenize,ner')

def lire_fichier(chemin, is_json=False):
    with open(chemin, encoding='utf-8') as f:
        return json.load(f) if is_json else f.read().strip()

def get_ent_dict(ent) -> dict:
    return {"mot": ent.text, "type": ent.type}

def remove_punctuation(token):
    return not re.match(r'[\W_]+', token)

# === 仅统计“整体语料”的处理时间；每个文本只跑一次 nlp ===
def process_corpus(file_paths, nlp):
    all_entites = []
    all_candidats = []
    all_tokens = []

    # 只测“整个语料”的墙钟时间（含 I/O、NER、tokenize）
    t0 = time.time()

    for file_path in file_paths:
        texte = lire_fichier(file_path)

        # 只调用一次 nlp
        doc = nlp(texte)

        # 实体（去重）
        seen = set()
        for ent in doc.ents:
            key = (ent.text, ent.type)
            if key not in seen:
                all_entites.append(get_ent_dict(ent))
                seen.add(key)

        # tokens
        tokens = [tok.text for sent in doc.sentences for tok in sent.tokens]
        all_tokens.extend(tokens)

        # 非实体候选
        ent_words = set(e["mot"] for e in all_entites)  # 注意：这是全局去重后的词集合
        for tok in tokens:
            if tok not in ent_words and remove_punctuation(tok):
                all_candidats.append(tok)

    total_time = time.time() - t0
    return all_entites, all_candidats, all_tokens, total_time

if __name__ == "__main__":
    file_paths = glob.glob("Corpus/*.txt")
    print("Fichiers à traiter :", len(file_paths))

    nlp = load_stanza_model(lang="fr")
    all_entites, all_candidats, all_tokens, total_time = process_corpus(file_paths, nlp)

    print("\n=== Résumé global (corpus entier) ===")
    print(f"Nombre total d'entités : {len(all_entites)}")
    print(f"Nombre total de candidats : {len(all_candidats)}")
    print(f"Nombre total de tokens : {len(all_tokens)}")
    print(f"Temps total (corpus) : {total_time:.2f} secondes")


INFO:stanza:Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


Fichiers à traiter : 10


INFO:stanza:Downloaded file to /root/stanza_resources/resources.json
INFO:stanza:Loading these models for language: fr (French):
| Processor | Package            |
----------------------------------
| tokenize  | combined           |
| mwt       | combined           |
| ner       | wikinergold_charlm |

INFO:stanza:Using device: cpu
INFO:stanza:Loading: tokenize
INFO:stanza:Loading: mwt
INFO:stanza:Loading: ner
INFO:stanza:Done loading processors!



=== Résumé global (corpus entier) ===
Nombre total d'entités : 8116
Nombre total de candidats : 227369
Nombre total de tokens : 296191
Temps total (corpus) : 1580.38 secondes
